# NLP project: attention heads and attention maps

Plots of the attention heads of the trained models in `results/`:

1. the mean attention map of every head,
2. attention maps of single sequences for the key heads,
3. previous-token and induction scores of every head,
4. how those scores develop over training,
5. the accuracy drop when each head is ablated,
6. the same views for the extended model,
7. head scores across model collapse generations, next to the heads in the discovered circuits.

Run `bash scripts/nlp_project_run.sh` first. The notebook picks the newest `results/<timestamp>_base` and `results/<timestamp>_extended` runs, and every collapse run that exists. Sections whose runs are missing are skipped. Figures are saved to `results/figures/nlp_project/`.

Use the `.venv` (or `.venv-circuit`) Python kernel. Only JAX is needed, not PyTorch.

In [ ]:
# Locate the project when launched from the root or a plotting subfolder.
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "main.py").is_file() and (p / "requirements.txt").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import glob
import json
from copy import deepcopy
from functools import partial

import numpy as np
import h5py as h5
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import jax
import jax.numpy as jnp
import equinox as eqx

from src import generations, main, main_utils, opto

%matplotlib inline

In [ ]:
base_folder = './results'
fig_folder = f'{base_folder}/figures/nlp_project'
os.makedirs(fig_folder, exist_ok=True)

N_SEQ = 1024                # real training sequences used for every plot
SEQ_SEED = 7                # the circuit discovery prompts use the same seed
N_CKPTS_OVER_TRAINING = 40  # checkpoints sampled for the over-training plots

def newest_run(name):
    runs = sorted(glob.glob(f'{base_folder}/*_{name}'))
    runs = [r for r in runs if os.path.isdir(r)]
    return runs[-1] if runs else None

BASE_RUN = newest_run('base')
EXTENDED_RUN = newest_run('extended')
print('base run:    ', BASE_RUN)
print('extended run:', EXTENDED_RUN)

In [ ]:
# Plot style: recessive axes, single-hue attention maps, a diverging map for signed scores.
matplotlib.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#8a8984', 'axes.labelcolor': '#52514e', 'xtick.color': '#52514e',
    'ytick.color': '#52514e', 'axes.grid': False, 'figure.dpi': 110, 'savefig.bbox': 'tight',
})
ATTN_CMAP = LinearSegmentedColormap.from_list('attn', ['#fcfcfb', '#9ec5f0', '#2a78d6', '#0d3a73'])
SCORE_CMAP = LinearSegmentedColormap.from_list('score', ['#eb6834', '#f3f2ee', '#2a78d6'])
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']  # categorical slots, fixed order
MUTED = '#c3c2b7'

def save(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(f'{fig_folder}/{name}.{ext}')

## Loading runs and extracting attention

A sequence of context length 2 is `x1 y1 x2 y2 xq`: exemplar and label pairs, then the query exemplar, whose label the model predicts. The extended model's sequences continue with the query label `yq` and the next symbol `xn`; it predicts the next symbol at `yq`.

The two head scores follow Singh et al. (2024):

* **previous-token score**: the attention each label position `yi` gives to its own exemplar `xi`, the step before it;
* **induction score**: the attention the query `xq` gives to the label of the matching exemplar, minus the attention it gives to the other label. A score of 1 means the head always looks up the correct label.

In [ ]:
class Run:
    """A trained run: its options, data, checkpoints and a fixed set of real sequences."""

    def __init__(self, folder, n_seq=N_SEQ, seed=SEQ_SEED):
        self.folder = folder
        argv, sys.argv = sys.argv, sys.argv[:1]  # the option parser reads sys.argv
        try:
            self.opts = main_utils.get_opts_from_json_file(f'{folder}/config.json')
        finally:
            sys.argv = argv
        self.data = main_utils.get_data_from_opts(self.opts)
        self.splits = main_utils.get_splits_from_opts(self.opts, self.data.shape)
        self.extended = self.opts.model_variant == 'extended'
        model = main_utils.get_model_from_opts(self.opts, (self.data.shape[-1],))
        opt_state = main_utils.get_optimizer_from_opts(self.opts).init(eqx.filter(model, eqx.is_array))
        key = jax.random.PRNGKey(0)
        self.ckpt_fmt = {'iter': -1, 'opt_state': opt_state, 'model': model,
                         'seeds': {'eval_model_seed': key, 'train_data_seed': key, 'train_model_seed': key}}
        sampler = main.make_train_data_sampler(self.opts, self.data, self.splits)
        self.seqs = generations.build_pool(sampler, n_seq, jax.random.PRNGKey(seed))
        self.n_ctx = self.seqs['labels'].shape[1] - (2 if self.extended else 1)
        self.tokens = self.token_names()

    def token_names(self):
        names = []
        for i in range(self.n_ctx):
            names += [f'x{i + 1}', f'y{i + 1}']
        names += ['xq', 'yq', 'xn'] if self.extended else ['xq']
        return names

    def checkpoints(self, gen_folder=None):
        folder = f'{gen_folder or self.folder}/checkpoints'
        files = sorted(os.listdir(folder))
        return [f'{folder}/{f}' for f in files], np.array([int(f.split('.')[0]) for f in files])

    def load(self, path=None):
        """Loads a checkpoint, by default the last one of the run."""
        path = path or self.checkpoints()[0][-1]
        return eqx.tree_deserialise_leaves(path, self.ckpt_fmt)['model']


@eqx.filter_jit
def _forward(model, x, y):
    keys = jax.random.split(jax.random.PRNGKey(0), x.shape[0])
    r = jax.vmap(partial(opto.default_model_fwd_fn, model=model))(x=x, y=y, key=keys)
    attn = jnp.stack([b['attn_output']['attn_scores'][:, 0] for b in r['transformer_output']['block_outputs']], 1)
    return attn, r['out']


def attention(run, model):
    """Attention patterns, shape (sequence, layer, head, query position, key position)."""
    attn, _ = _forward(model, run.seqs['examples'], run.seqs['labels'])
    return np.asarray(attn)


def head_scores(run, attn):
    """Previous-token and induction score of every head, each shape (layer, head)."""
    n = run.n_ctx
    labels_pos = 2 * np.arange(n) + 1
    prev = attn[:, :, :, labels_pos, labels_pos - 1].mean(axis=(0, 3))
    s = np.asarray(run.seqs['symbols'])
    match = s[:, :n] == s[:, [n]]               # which context items match the query
    keep = match.any(1) & (~match).any(1)       # sequences with a correct and a wrong label
    q = attn[keep][:, :, :, 2 * n, labels_pos]  # query attention to every context label
    m = match[keep][:, None, None, :]
    correct = (q * m).sum(-1) / m.sum(-1)
    wrong = (q * ~m).sum(-1) / (~m).sum(-1)
    return prev, (correct - wrong).mean(0)


runs = {}
if BASE_RUN:
    runs['base'] = Run(BASE_RUN)
if EXTENDED_RUN:
    runs['extended'] = Run(EXTENDED_RUN)

## 1. Mean attention map of every head

Each panel is one head: rows are query positions, columns the positions they attend to, averaged over the real sequences. The model is causal, so the upper triangle is empty.

In [ ]:
def plot_attention_grid(run, attn, title, name):
    mean = attn.mean(0)
    n_layers, n_heads = mean.shape[:2]
    fig, axs = plt.subplots(n_layers, n_heads, figsize=(1.9 * n_heads, 2.1 * n_layers + 0.6), squeeze=False)
    for l in range(n_layers):
        for h in range(n_heads):
            ax = axs[l, h]
            im = ax.imshow(mean[l, h], cmap=ATTN_CMAP, vmin=0, vmax=1)
            ax.set_xticks(range(len(run.tokens)), run.tokens, fontsize=8)
            ax.set_yticks(range(len(run.tokens)), run.tokens if h == 0 else [], fontsize=8)
            ax.tick_params(length=0)
            ax.set_title(f'A{l}.{h}', fontsize=10)
            for side in ax.spines.values():
                side.set_visible(False)
    fig.suptitle(title)
    fig.colorbar(im, ax=axs, fraction=0.015, pad=0.01, label='attention')
    save(fig, name)
    plt.show()

if 'base' in runs:
    run = runs['base']
    base_model = run.load()
    base_attn = attention(run, base_model)
    plot_attention_grid(run, base_attn, 'Base model: mean attention of every head (rows: layer 0, layer 1)',
                        'base_attention_maps')

## 2. Head scores

Layer 0 should contain previous-token heads and layer 1 induction heads. The heads with the highest score in each layer are used in the following plots.

In [ ]:
def plot_head_scores(prev, ind, title, name):
    n_layers, n_heads = prev.shape
    fig, axs = plt.subplots(1, 2, figsize=(12, 1.4 + 0.7 * n_layers), layout='constrained')
    for ax, scores, label, cmap, lims in [
            (axs[0], prev, 'previous-token score', ATTN_CMAP, (0, 1)),
            (axs[1], ind, 'induction score', SCORE_CMAP, (-1, 1))]:
        im = ax.imshow(scores, cmap=cmap, vmin=lims[0], vmax=lims[1], aspect='auto')
        for l in range(n_layers):
            for h in range(n_heads):
                v = scores[l, h]
                ax.text(h, l, f'{v:.2f}', ha='center', va='center', fontsize=9,
                        color='white' if abs(v) > 0.6 else '#0b0b0b')
        ax.set_xticks(range(n_heads), [f'H{h}' for h in range(n_heads)])
        ax.set_yticks(range(n_layers), [f'layer {l}' for l in range(n_layers)])
        ax.tick_params(length=0)
        for side in ax.spines.values():
            side.set_visible(False)
        ax.set_title(label)
        fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    fig.suptitle(title)
    save(fig, name)
    plt.show()

def key_heads(prev, ind):
    """(layer, head) of the strongest previous-token head in layer 0 and induction head in the last layer."""
    return (0, int(np.argmax(prev[0]))), (prev.shape[0] - 1, int(np.argmax(ind[-1])))

if 'base' in runs:
    base_prev, base_ind = head_scores(runs['base'], base_attn)
    plot_head_scores(base_prev, base_ind, 'Base model: head scores', 'base_head_scores')
    BASE_PREV_HEAD, BASE_IND_HEAD = key_heads(base_prev, base_ind)
    print('strongest previous-token head: A{}.{}'.format(*BASE_PREV_HEAD))
    print('strongest induction head:      A{}.{}'.format(*BASE_IND_HEAD))

## 3. Attention maps of single sequences

The key heads on a few sequences. The labels above each map give each exemplar's class and label; the matching exemplar shares the query's class.

In [ ]:
def plot_single_sequences(run, attn, heads, title, name, n_examples=3):
    s, y = np.asarray(run.seqs['symbols']), np.asarray(run.seqs['labels'])
    n = run.n_ctx
    match = s[:, :n] == s[:, [n]]
    examples = np.flatnonzero(match.any(1) & (~match).any(1))[:n_examples]
    fig, axs = plt.subplots(len(heads), len(examples), figsize=(3.3 * len(examples), 3.2 * len(heads)), squeeze=False)
    for col, i in enumerate(examples):
        desc = ', '.join(f'x{j + 1}=c{s[i, j]}/y{j + 1}={y[i, j]}' for j in range(n))
        for row, (l, h) in enumerate(heads):
            ax = axs[row, col]
            im = ax.imshow(attn[i, l, h], cmap=ATTN_CMAP, vmin=0, vmax=1)
            ax.set_xticks(range(len(run.tokens)), run.tokens, fontsize=8)
            ax.set_yticks(range(len(run.tokens)), run.tokens, fontsize=8)
            ax.tick_params(length=0)
            ax.set_title(f'A{l}.{h}  seq {i}' + (f'\n{desc}, xq=c{s[i, n]}' if row == 0 else ''), fontsize=8)
            for side in ax.spines.values():
                side.set_visible(False)
    fig.suptitle(title)
    fig.colorbar(im, ax=axs, fraction=0.02, pad=0.02, label='attention')
    save(fig, name)
    plt.show()

if 'base' in runs:
    plot_single_sequences(runs['base'], base_attn, [BASE_PREV_HEAD, BASE_IND_HEAD],
                          'Base model: previous-token head (top) and induction head (bottom)', 'base_single_sequences')

## 4. Head scores over training

Every head's score at checkpoints across training. The strongest heads are coloured and the rest grey. Induction heads appear abruptly, together with the drop in loss.

In [ ]:
def scores_over_training(run, n_ckpts=N_CKPTS_OVER_TRAINING, gen_folder=None):
    paths, iters = run.checkpoints(gen_folder)
    idx = np.unique(np.round(np.linspace(0, len(paths) - 1, min(n_ckpts, len(paths)))).astype(int))
    prev, ind = [], []
    for i in idx:
        p, s = head_scores(run, attention(run, run.load(paths[i])))
        prev.append(p)
        ind.append(s)
    return iters[idx], np.stack(prev), np.stack(ind)

def plot_over_training(iters, series, layer, label, title, ax, highlight=2):
    final = series[-1, layer]
    top = list(np.argsort(final)[::-1][:highlight])
    for h in range(series.shape[2]):
        if h not in top:
            ax.plot(iters, series[:, layer, h], color=MUTED, lw=1.2, zorder=1)
    for rank, h in enumerate(top):
        ax.plot(iters, series[:, layer, h], color=SERIES[rank], lw=2, zorder=2, label=f'A{layer}.{h}')
        ax.annotate(f'A{layer}.{h}', (iters[-1], series[-1, layer, h]), xytext=(4, 0),
                    textcoords='offset points', va='center', fontsize=9, color='#0b0b0b')
    ax.plot([], [], color=MUTED, lw=1.2, label='other heads')
    ax.set_xlabel('# sequences seen')
    ax.set_ylabel(label)
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=9)
    ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))

def plot_training_figure(run, name, title):
    iters, prev, ind = scores_over_training(run)
    with h5.File(f'{run.folder}/log.h5', 'r') as f:
        eval_iters, loss = f['eval_iter'][:], np.mean(f['fsl_train']['loss'][:], axis=1)
    fig, axs = plt.subplots(1, 3, figsize=(16, 4.2))
    axs[0].plot(eval_iters, loss, color=SERIES[0], lw=2)
    axs[0].axhline(np.log(2), color='#52514e', ls='--', lw=1)
    axs[0].annotate('random label from context', (eval_iters[-1], np.log(2)), xytext=(0, 4),
                    textcoords='offset points', ha='right', fontsize=9, color='#52514e')
    axs[0].set_xlabel('# sequences seen')
    axs[0].set_ylabel('training loss')
    axs[0].set_title('loss')
    axs[0].ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
    plot_over_training(iters, prev, 0, 'previous-token score', 'layer 0 previous-token score', axs[1])
    plot_over_training(iters, ind, prev.shape[1] - 1, 'induction score', 'last-layer induction score', axs[2])
    fig.suptitle(title)
    fig.tight_layout()
    save(fig, name)
    plt.show()
    return iters, prev, ind

if 'base' in runs:
    base_training = plot_training_figure(runs['base'], 'base_scores_over_training', 'Base model: head formation during training')

## 5. Head ablations

Accuracy on the real sequences when one head's output is ablated (its values set to zero), next to each head's score. A large drop marks a head the model depends on.

In [ ]:
def ablation_accuracy(run, model, heads):
    n = run.n_ctx
    x, y = run.seqs['examples'], run.seqs['labels']
    keys = jax.random.split(jax.random.PRNGKey(0), x.shape[0])
    def acc(head):
        o = deepcopy(run.opts)
        o.opto_ablate_heads = None if head is None else ['{}:{}'.format(*head)]
        fn = jax.jit(jax.vmap(partial(opto.make_fn_from_opts(o), model)))
        out = fn(x, y, keys)['out']
        return float(jnp.mean(jnp.argmax(out[:, n], -1) == y[:, n]))
    return acc(None), {head: acc(head) for head in heads}

def plot_ablations(run, model, prev, ind, title, name):
    n_layers, n_heads = prev.shape
    heads = [(l, h) for l in range(n_layers) for h in range(n_heads)]
    full, accs = ablation_accuracy(run, model, heads)
    fig, axs = plt.subplots(1, n_layers, figsize=(6.5 * n_layers, 3.8), sharey=True, squeeze=False)
    for l in range(n_layers):
        ax = axs[0, l]
        score = prev[l] if l == 0 else ind[l]
        drops = [full - accs[(l, h)] for h in range(n_heads)]
        order = np.argsort(score)[::-1]
        ax.bar(range(n_heads), [drops[h] for h in order], color=SERIES[0], width=0.6)
        ax.set_xticks(range(n_heads), [f'A{l}.{h}\n{score[h]:.2f}' for h in order], fontsize=8)
        ax.axhline(0, color='#8a8984', lw=0.8)
        ax.set_title(f'layer {l}, heads sorted by {"previous-token" if l == 0 else "induction"} score')
        ax.set_ylabel('accuracy drop when ablated' if l == 0 else '')
    fig.suptitle(f'{title} (full model accuracy {full:.3f})')
    fig.tight_layout()
    save(fig, name)
    plt.show()

if 'base' in runs:
    plot_ablations(runs['base'], base_model, base_prev, base_ind, 'Base model: single-head ablations', 'base_head_ablations')

## 6. Extended model

The extended model also predicts the next symbol at `yq`. Its attention maps include the query label and the next symbol, and its value prediction at `xq` can be compared with the base model's.

In [ ]:
if 'extended' in runs:
    run = runs['extended']
    ext_model = run.load()
    ext_attn = attention(run, ext_model)
    plot_attention_grid(run, ext_attn, 'Extended model: mean attention of every head', 'extended_attention_maps')
    ext_prev, ext_ind = head_scores(run, ext_attn)
    plot_head_scores(ext_prev, ext_ind, 'Extended model: head scores', 'extended_head_scores')
    EXT_PREV_HEAD, EXT_IND_HEAD = key_heads(ext_prev, ext_ind)
    plot_single_sequences(run, ext_attn, [EXT_PREV_HEAD, EXT_IND_HEAD],
                          'Extended model: previous-token head (top) and induction head (bottom)',
                          'extended_single_sequences')

In [ ]:
if 'extended' in runs:
    # Where the next-symbol prediction looks: attention from yq, averaged over sequences
    run = runs['extended']
    yq = run.tokens.index('yq')
    from_yq = ext_attn[:, :, :, yq, :yq + 1].mean(0)
    n_layers, n_heads = from_yq.shape[:2]
    fig, axs = plt.subplots(1, n_layers, figsize=(6 * n_layers, 3.8), squeeze=False, layout='constrained')
    for l in range(n_layers):
        ax = axs[0, l]
        im = ax.imshow(from_yq[l], cmap=ATTN_CMAP, vmin=0, vmax=1, aspect='auto')
        ax.set_xticks(range(yq + 1), run.tokens[:yq + 1])
        ax.set_yticks(range(n_heads), [f'A{l}.{h}' for h in range(n_heads)])
        ax.tick_params(length=0)
        ax.set_title(f'layer {l}: attention from yq')
        for side in ax.spines.values():
            side.set_visible(False)
    fig.colorbar(im, ax=axs, fraction=0.02, pad=0.02, label='attention')
    fig.suptitle('Extended model: attention at the next-symbol prediction')
    save(fig, 'extended_attention_from_yq')
    plt.show()

In [ ]:
if 'extended' in runs:
    ext_training = plot_training_figure(runs['extended'], 'extended_scores_over_training',
                                        'Extended model: head formation during training')
    plot_ablations(runs['extended'], ext_model, ext_prev, ext_ind,
                   'Extended model: single-head ablations (value prediction)', 'extended_head_ablations')

## 7. Heads across model collapse generations

For each collapse run, every generation's final model is analysed on the same real sequences. The heatmaps show how the previous-token and induction scores of each head change from generation 0 (real data) to the last generation. Where circuit discovery ran, the right panel marks the heads that appear in the ACDC circuit of each generation for the exemplar-swap target.

In [ ]:
def generation_folders(run_folder):
    gens = sorted(glob.glob(f'{run_folder}/gen_*'), key=lambda p: int(p.rsplit('_', 1)[1]))
    return [g for g in gens if os.path.isdir(f'{g}/checkpoints') and os.listdir(f'{g}/checkpoints')]

def acdc_heads(gen_folder, n_layers, n_heads, target='value_exemplar_swap'):
    """1 for every head with an edge in the ACDC circuit, or None without circuit outputs."""
    path = f'{gen_folder}/circuit/{target}_acdc_edges.json'
    if not os.path.exists(path):
        return None
    used = np.zeros((n_layers, n_heads))
    for e in json.load(open(path)):
        if e['in_circuit']:
            for node in (e['src'], e['dest']):
                if node.startswith('A'):
                    l, h = node[1:].split('.')[:2]
                    used[int(l), int(h)] = 1
    return used

def plot_collapse_run(run_folder):
    gens = generation_folders(run_folder)
    if len(gens) < 2:
        return
    run = Run(run_folder)
    prev, ind, circ = [], [], []
    for g in gens:
        model = run.load(run.checkpoints(g)[0][-1])
        p, s = head_scores(run, attention(run, model))
        prev.append(p)
        ind.append(s)
        circ.append(acdc_heads(g, *p.shape))
    prev, ind = np.stack(prev), np.stack(ind)
    has_circ = all(c is not None for c in circ)
    n_heads = prev.shape[2]
    panels = [(prev[:, 0], 'layer 0 previous-token score', ATTN_CMAP, (0, 1)),
              (ind[:, -1], 'last-layer induction score', SCORE_CMAP, (-1, 1))]
    if has_circ:
        panels.append((np.stack([c[-1] for c in circ]), 'last-layer head in ACDC circuit', ATTN_CMAP, (0, 1)))
    fig, axs = plt.subplots(1, len(panels), figsize=(5.2 * len(panels), 0.9 + 0.45 * len(gens)))
    for ax, (values, label, cmap, lims) in zip(axs, panels):
        ax.imshow(values, cmap=cmap, vmin=lims[0], vmax=lims[1], aspect='auto')
        for g in range(values.shape[0]):
            for h in range(n_heads):
                ax.text(h, g, f'{values[g, h]:.2f}' if 'circuit' not in label else ('in' if values[g, h] else ''),
                        ha='center', va='center', fontsize=8,
                        color='white' if abs(values[g, h]) > 0.6 else '#0b0b0b')
        ax.set_xticks(range(n_heads), [f'H{h}' for h in range(n_heads)])
        ax.set_yticks(range(len(gens)), [f'gen {g}' for g in range(len(gens))])
        ax.tick_params(length=0)
        for side in ax.spines.values():
            side.set_visible(False)
        ax.set_title(label)
    name = os.path.basename(run_folder).split('_', 1)[1]
    fig.suptitle(f'{name}: head scores per generation')
    fig.tight_layout()
    save(fig, f'{name}_heads_per_generation')
    plt.show()

collapse_runs = sorted(r for r in glob.glob(f'{base_folder}/*_collapse_*') if os.path.isdir(r))
print(f'{len(collapse_runs)} collapse runs found')
for r in collapse_runs:
    plot_collapse_run(r)

## 8. Induction score of the strongest head across conditions

One line per collapse condition of the main experiments: the largest last-layer induction score of each generation. The control (`real_only`) trains on real data in every generation.

In [ ]:
def strongest_induction_per_generation(run_folder):
    run = Run(run_folder)
    return [float(head_scores(run, attention(run, run.load(run.checkpoints(g)[0][-1])))[1][-1].max())
            for g in generation_folders(run_folder)]

CONDITIONS = ['real_only', 'replace50', 'replace90', 'replace100', 'accumulate']
for variant in ['base', 'extended']:
    found = {c: newest_run(f'collapse_{variant}_{c}') for c in CONDITIONS}
    found = {c: r for c, r in found.items() if r and len(generation_folders(r)) > 1}
    if not found:
        continue
    fig, ax = plt.subplots(figsize=(8, 4.2))
    colours = dict(zip(CONDITIONS, [MUTED] + SERIES))  # the control is grey
    for c, r in found.items():
        values = strongest_induction_per_generation(r)
        ax.plot(range(len(values)), values, color=colours[c], lw=2, marker='o', ms=8,
                markeredgecolor='white', markeredgewidth=2, label=c)
    ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
    ax.set_xlabel('generation')
    ax.set_ylabel('strongest induction score')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f'{variant} model: induction head strength across generations')
    ax.legend(frameon=False, fontsize=9)
    save(fig, f'collapse_{variant}_induction_across_generations')
    plt.show()